# RU Llama 3 Baseline and QLoRA

Main thesis notebook for the Russian branch. It uses the same base model as the English branch: `meta-llama/Meta-Llama-3-8B-Instruct`.

## 1. Install dependencies

Run this in Google Colab with a GPU runtime.

In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets scipy

## 2. Upload repository files

Upload the repository or mount Google Drive so that `src/`, `configs/`, and `data/processed/ru/` are available in the working directory.

In [ ]:
import getpass
import os
from pathlib import Path

assert Path('src').exists(), 'Run %cd to the project root before continuing.'
assert Path('data/processed/ru/eval_payload.jsonl').exists(), 'RU eval payload is missing.'

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

!huggingface-cli whoami

## 3. Baseline generation

In [ ]:
!PYTHONPATH=$PWD python -m src.generation.generate_baseline \
  --language ru \
  --eval-jsonl data/processed/ru/eval_payload.jsonl \
  --output-csv outputs/ru/baseline_generations.csv \
  --max-samples 100

## 4. QLoRA training

In [ ]:
!PYTHONPATH=$PWD python -m src.training.train_qlora \
  --train-jsonl data/processed/ru/train_payload.jsonl \
  --eval-jsonl data/processed/ru/eval_payload.jsonl \
  --adapter-dir adapters/ru_lora_adapter \
  --max-train-samples 500 \
  --max-eval-samples 100

## 5. Fine-tuned generation

In [ ]:
!PYTHONPATH=$PWD python -m src.generation.generate_with_adapter \
  --language ru \
  --eval-jsonl data/processed/ru/eval_payload.jsonl \
  --adapter-dir adapters/ru_lora_adapter \
  --output-csv outputs/ru/finetuned_generations.csv \
  --max-samples 100

## 6. Stylometry and statistics

In [ ]:
!PYTHONPATH=$PWD python -m src.evaluation.run_stylometry \
  --language ru \
  --human-jsonl data/processed/ru/eval_payload.jsonl \
  --baseline-csv outputs/ru/baseline_generations.csv \
  --finetuned-csv outputs/ru/finetuned_generations.csv \
  --features-csv outputs/ru/stylometric_features.csv \
  --summary-csv outputs/ru/metric_summary.csv